In [21]:
## Librerias ##
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
listings = pd.read_csv(
    "data/Listings.csv", encoding = "ISO-8859-1", 
    low_memory=False,
    parse_dates =["host_since"]
    )

In [3]:
## informacion general del csv listings.csv ##

listings.head()
listings.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 279712 entries, 0 to 279711
Data columns (total 33 columns):
 #   Column                       Non-Null Count   Dtype         
---  ------                       --------------   -----         
 0   listing_id                   279712 non-null  int64         
 1   name                         279537 non-null  object        
 2   host_id                      279712 non-null  int64         
 3   host_since                   279547 non-null  datetime64[ns]
 4   host_location                278872 non-null  object        
 5   host_response_time           150930 non-null  object        
 6   host_response_rate           150930 non-null  float64       
 7   host_acceptance_rate         166625 non-null  float64       
 8   host_is_superhost            279547 non-null  object        
 9   host_total_listings_count    279547 non-null  float64       
 10  host_has_profile_pic         279547 non-null  object        
 11  host_identity_verified    

In [4]:
## quitamos columnas innecesarias ##

columnas_eliminar = [
    "name", "host_id", "host_location", "host_response_time", "district",
    "host_response_rate", "host_acceptance_rate", "host_is_superhost", "host_has_profile_pic",
    "host_identity_verified", "amenities", "review_scores_accuracy",
    "review_scores_checkin", "review_scores_communication","instant_bookable"
    ]
listings_reduce = listings.drop(columns=columnas_eliminar, errors="ignore")
print("columnas restantes:",listings_reduce.shape[1])

columnas restantes: 18


In [5]:
## vemos la cantidad de celdas vacias que contine cada columna ##
listings_reduce.isna().sum()

listing_id                       0
host_since                     165
host_total_listings_count      165
neighbourhood                    0
city                             0
latitude                         0
longitude                        0
property_type                    0
room_type                        0
accommodates                     0
bedrooms                     29435
price                            0
minimum_nights                   0
maximum_nights                   0
review_scores_rating         91405
review_scores_cleanliness    91665
review_scores_location       91775
review_scores_value          91785
dtype: int64

In [6]:
## Elimina las filas (no columnas) donde host_since o host_total_listings_count estén vacías. ##
listings_reduce = listings_reduce.dropna(subset=["host_since", "host_total_listings_count"])


In [7]:
## Crea una columna nueva con True si el anuncio tiene valoración global y False si está vacía. ##
listings_reduce["has_reviews"] = listings_reduce["review_scores_rating"].notna()

print(listings_reduce.shape)

(279547, 19)


In [8]:
# Nulos de bedrooms por ciudad #
(listings_reduce.groupby("city")["bedrooms"]
    .agg(nulos=lambda s: s.isna().sum(), pct=lambda s: s.isna().mean().round(3), total="size")
    .sort_values("pct", ascending=False))

,nulos,pct,total
city,,,
Paris,13394,0.207,64657
Hong Kong,1230,0.174,7083
Bangkok,2142,0.111,19360
New York,3608,0.098,36994
Istanbul,2034,0.083,24505
Cape Town,1379,0.072,19070
Sydney,2201,0.066,33596
Rio de Janeiro,1746,0.066,26591
Mexico City,809,0.040,20055


In [9]:
# ¿Los anuncios sin bedrooms son más pequeños? (si es así, probablemente son estudios)
listings_reduce.groupby(listings_reduce["bedrooms"].isna())["accommodates"].describe()

,count,mean,std,min,25%,50%,75%,max
bedrooms,,,,,,,,
False,250132.0,3.400708,2.192104,1.0,2.0,3.0,4.0,16.0
True,29415.0,2.336529,1.175463,0.0,2.0,2.0,2.0,16.0


In [10]:
# ¿Existen anuncios con 0 dormitorios? (así se marcan los estudios cuando sí se informa)
listings_reduce["bedrooms"].value_counts().sort_index().head(8)

bedrooms
1.0    170069
2.0     51355
3.0     18505
4.0      6578
5.0      2103
6.0       701
7.0       246
8.0       124
Name: count, dtype: int64

In [11]:
# Marcar qué filas se imputan
listings_reduce["bedrooms_imputed"] = listings_reduce["bedrooms"].isna()

# 1º intento: mediana por ciudad + tipo de alojamiento + capacidad
med = listings_reduce.groupby(["city", "room_type", "accommodates"])["bedrooms"].transform("median")
listings_reduce["bedrooms"] = listings_reduce["bedrooms"].fillna(med)

# 2º intento (grupos sin datos): ciudad + capacidad
med2 = listings_reduce.groupby(["city", "accommodates"])["bedrooms"].transform("median")
listings_reduce["bedrooms"] = listings_reduce["bedrooms"].fillna(med2)

# Último recurso: mediana global
listings_reduce["bedrooms"] = listings_reduce["bedrooms"].fillna(listings_reduce["bedrooms"].median())

# Medianas de .5 -> entero
listings_reduce["bedrooms"] = listings_reduce["bedrooms"].round().astype(int)

print(listings_reduce["bedrooms"].isna().sum(), "nulos restantes")
print(listings_reduce["bedrooms_imputed"].sum(), "filas imputadas")

0 nulos restantes
29415 filas imputadas


In [22]:
# 1. Eliminar precios 0 (113 filas)
listings_reduce = listings_reduce[listings_reduce["price"] > 0].copy()

# 2. Límites por ciudad: IQR sobre log10(precio), factor 3 = atípicos extremos
log_p = np.log10(listings_reduce["price"])
q = log_p.groupby(listings_reduce["city"]).quantile([.25, .75]).unstack()
iqr = q[0.75] - q[0.25]

lim = pd.DataFrame({
    "lim_inf": 10 ** (q[0.25] - 3 * iqr),
    "lim_sup": 10 ** (q[0.75] + 3 * iqr),
})

# 3. Marcar atípicos (sin borrar todavía)
aux = listings_reduce.join(lim, on="city")
listings_reduce["price_outlier"] = (aux["price"] > aux["lim_sup"]) | (aux["price"] < aux["lim_inf"])

# 4. Revisar
print(lim.round(0))
print(listings_reduce.groupby("city")["price_outlier"].agg(["sum", "mean"]).round(4))

                lim_inf  lim_sup
city                            
Bangkok            35.0 37,994.0
Cape Town          17.0 85,300.0
Hong Kong          11.0 15,837.0
Istanbul            6.0 12,042.0
Mexico City        17.0 26,498.0
New York            4.0  2,407.0
Paris               7.0  1,010.0
Rio de Janeiro      4.0 23,646.0
Rome                4.0  1,054.0
Sydney              3.0  5,677.0
                sum  mean
city                     
Bangkok          80   0.0
Cape Town        22   0.0
Hong Kong        12   0.0
Istanbul         53   0.0
Mexico City      37   0.0
New York         62   0.0
Paris           192   0.0
Rio de Janeiro   51   0.0
Rome            104   0.0
Sydney           28   0.0


In [23]:
# Quitar precios 0 y atípicos extremos (ya marcados en price_outlier)
listings_reduce = listings_reduce[~listings_reduce["price_outlier"]].copy()
listings_reduce = listings_reduce.drop(columns="price_outlier")

print(listings_reduce.shape)

(278793, 20)
